# Map2Decibel
### Street-level road noise prediction from OpenStreetMap morphology

**Paper:** *Maps Are Loud: A Scalable And Quick Estimation Approach To Traffic Noise Pollution* (under review)

This notebook reproduces the core prediction pipeline from the paper.
Given any city name or coordinates, it:
1. Downloads road network geometry from OpenStreetMap
2. Extracts 20 morphological features per road segment
3. Applies the unified RF model to predict L_den (dBA)
4. Renders an interactive map coloured by predicted noise

**No OpeNoise simulation or traffic data required.**

---
**Setup:** Run the cell below to install dependencies, then proceed.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CMU-SALUS-Lab/Map2Decibel/blob/main/map2decibel_demo.ipynb)

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# Comment out if already installed locally
%pip install osmnx geopandas scikit-learn joblib folium matplotlib scipy -q

In [ ]:
import os, warnings, numpy as np
import geopandas as gpd
import osmnx as ox
import joblib
import folium
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import rankdata
warnings.filterwarnings('ignore')

print('All imports OK')

## Step 1 — Load the unified models

Download `unified_norm_rf.joblib` and `unified_raw_rf.joblib` from the
[releases page](https://github.com/CMU-SALUS-Lab/Map2Decibel/releases) and
place them in the same folder as this notebook.

| Model | Output | Calibration | Use for |
|---|---|---|---|
| `unified_norm_rf.joblib` | Noise rank (z-score) | Linear rank→dB per city | Relative ranking within a city |
| `unified_raw_rf.joblib` | Direct dBA | None | Absolute estimates (may be biased outside Europe) |

In [ ]:
MODELS = {}
for name, fname in [('norm', 'unified_norm_rf.joblib'),
                     ('raw',  'unified_raw_rf.joblib')]:
    if os.path.exists(fname):
        b = joblib.load(fname)
        MODELS[name] = {'model': b['model'],
                        'feat_cols': b.get('feature_cols', [])}
        print(f'Loaded {name} model (trained on {len(b.get("cities",[]))} cities)')
    else:
        print(f'NOT FOUND: {fname} — download from the releases page')

assert MODELS, 'No models loaded — check the paths above'

## Step 2 — Choose a city

Set `CITY_NAME` **or** `LAT, LON`. City name is geocoded automatically.
Adjust `RADIUS_M` to control the study area size (500–3000 m recommended).

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────
CITY_NAME = 'Singapore'   # or set to '' and fill LAT/LON below
LAT, LON  = 0.0, 0.0     # ignored if CITY_NAME is set
RADIUS_M  = 2000          # study area radius in metres
# ─────────────────────────────────────────────────────────────────────────────

if CITY_NAME.strip():
    pt = ox.geocode_to_gdf(CITY_NAME).geometry.centroid.iloc[0]
    LAT, LON = pt.y, pt.x
    print(f'Geocoded: {CITY_NAME} → ({LAT:.4f}, {LON:.4f})')
else:
    print(f'Using coordinates: ({LAT}, {LON})')

## Step 3 — Download OSM data and extract features

In [ ]:
# Download road network, buildings, vegetation, water
print(f'Downloading OSM data ({RADIUS_M} m radius)...')
G         = ox.graph_from_point((LAT, LON), dist=RADIUS_M,
                                 network_type='all', retain_all=True)
edges     = ox.graph_to_gdfs(G, nodes=False).reset_index()
buildings = ox.features_from_point((LAT, LON), dist=RADIUS_M,
                                    tags={'building': True})
green     = ox.features_from_point((LAT, LON), dist=RADIUS_M,
                                    tags={'natural': ['wood','scrub','grass'],
                                          'landuse': ['forest','grass','meadow']})
water     = ox.features_from_point((LAT, LON), dist=RADIUS_M,
                                    tags={'natural': 'water', 'waterway': True})

zone  = int((LON + 180) / 6) + 1
hemi  = '6' if LAT >= 0 else '7'
CRS_M = f'EPSG:32{hemi}{zone:02d}'

print(f'{len(edges):,} road segments | {len(buildings):,} buildings | CRS: {CRS_M}')

In [ ]:
# Extract 20 morphological features
# This calls the same function used in the paper pipeline
from noisy_feature_extraction_v1 import extract_features

edges_m, feat_cols = extract_features(edges, buildings, green, water, G, CRS_M)
print(f'Feature extraction complete: {len(feat_cols)} features, {len(edges_m):,} segments')

## Step 4 — Predict noise

In [ ]:
# Norm model: z-normalise within city → noise rank + percentile
if 'norm' in MODELS:
    m     = MODELS['norm']
    avail = [f for f in m['feat_cols'] if f in edges_m.columns]
    X     = edges_m[avail].fillna(0).values.astype(float)
    X_z   = (X - X.mean(0)) / (X.std(0) + 1e-9)
    scores = m['model'].predict(X_z)
    edges_m['noise_rank'] = scores
    edges_m['noise_pct']  = (rankdata(scores) / len(scores) * 100).round(1)
    print(f'Norm model: rank range {scores.min():.2f} – {scores.max():.2f}')

# Raw model: predict absolute dB directly
if 'raw' in MODELS:
    m     = MODELS['raw']
    avail = [f for f in m['feat_cols'] if f in edges_m.columns]
    X     = edges_m[avail].fillna(0).values.astype(float)
    raw_db = m['model'].predict(X)
    edges_m['noise_raw_db'] = raw_db.round(1)
    print(f'Raw model:  dB range {raw_db.min():.1f} – {raw_db.max():.1f} dBA')

## Step 5 — Interactive map

Change `DISPLAY_COL` to switch between raw dB and percentile rank.

In [ ]:
DISPLAY_COL = 'noise_raw_db'   # or 'noise_pct'

gdf  = edges_m.to_crs('EPSG:4326')
cmap = plt.cm.RdYlBu_r
norm = mcolors.Normalize(vmin=0, vmax=99)

m = folium.Map(location=[LAT, LON], zoom_start=14,
               tiles='CartoDB positron')

hw_lw = {'motorway':5,'trunk':5,'primary':4,'secondary':3,'tertiary':2}

for _, row in gdf[gdf[DISPLAY_COL].notna()].iterrows():
    try:
        v      = float(row[DISPLAY_COL])
        color  = mcolors.to_hex(cmap(norm(np.clip(v, 0, 99))))
        hw     = str(row.get('highway', ''))
        weight = next((w for k,w in hw_lw.items() if k in hw), 1.5)
        coords = [[c[1],c[0]] for c in row.geometry.coords]
        folium.PolyLine(coords, color=color, weight=weight, opacity=0.9,
                        tooltip=f'{v:.1f} dBA').add_to(m)
    except Exception:
        continue

m

## Step 6 — Export results (optional)

Save predictions as a GeoPackage for use in QGIS or further analysis.

In [ ]:
city_slug = CITY_NAME.lower().replace(' ', '_') or 'custom'
out_path  = f'noise_prediction_{city_slug}.gpkg'

save_cols = ['geometry', 'highway'] + \
            [c for c in ['noise_raw_db','noise_pct','noise_rank']
             if c in gdf.columns]

gdf[save_cols].to_file(out_path, driver='GPKG')
print(f'Saved: {out_path}')
print('Load in QGIS: Layer → Add Layer → Add Vector Layer')

---
## Notes

**Raw dB interpretation:**
The raw model predicts absolute L_den without city-specific calibration.
It may be systematically biased outside European contexts (see paper §3.4).
Use `noise_pct` (percentile rank) for relative within-city comparisons.

**OpeNoise calibration (optional):**
If you run OpeNoise for a small tile of your city, you can fit a linear
calibration `dB = a * noise_rank + b` to anchor the norm model to local
absolute values. See `validate_prediction.py` in the repository.

**Citation:**
```
[Author(s) withheld for blind review] (2025).
Maps Are Loud: A Scalable And Quick Estimation Approach
To Traffic Noise Pollution. Under review.
Code: https://anonymous.4open.science/r/[REPO-ID]
```

**Reproducibility:**
Full training pipeline, feature extraction code, and per-city model files
are available in the repository. See `noise_proxy_pipeline_v5.py` and
`multi_city_model_v2.py`.

*Traffic count and speed inputs are assumed from EU highway-class defaults
(EU Good Practice Guide 2.5). Not suitable for regulatory noise assessment.*